# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAyyanHassan/flyrank-ml-internship-work/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / Scoring.**

My lane (Refresh / Content Opportunity Scoring, chosen in ML-02) is not "will this page decline?"
in isolation — it's "of 30,000 pages, which ones does the SEO content lead look at *first* this
week?" That's a "which ones first?" question, and per the framing skill's own mapping table,
that maps to **ranking/scoring**, with a priority score as the target and precision@K as the
metric — not classification.

The distinction matters in practice: a classifier alone would give me a yes/no label per page
("declining" or "not"), but a review queue with limited editor time doesn't need a verdict on
all 30,000 pages — it needs an ordered list, so the top 20-50 are worth an editor's time. Under
the hood I can still use a classifier's *probability output* as the ranking score (that's exactly
what the repo's own pipeline does in `03_train_model.py` — it ranks by `predict_proba`, not by
the binary 0/1 prediction). So the task type is ranking; classification is the mechanism, not
the goal.

In [ ]:
# No computation needed here -- Section 1 is the framing itself, argued in the markdown above.
# The task-type choice is checked against real data below (Sections 3-4).

: 

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label`**, defined in the starter pipeline as `trend_direction == "down"`.

Being honest about what this actually is, per the framing skill's own rule ("the target must be
observed, not defined"): this is a **proxy label**, not a true future outcome. `trend_direction`
is itself computed from `trend_pct`, which compares the last 30 days of impressions against the
prior 30 days — both windows are already in the past relative to when I'd compute it. It tells me
"this page has recently been declining," not "this page will decline next month." That's a real
limitation I'm flagging now, not hiding.

A stronger version for the capstone (per `docs/ml-intern-dataset-and-lane-guide.md`, section 5)
would define the label on a genuinely future window: features from the prior 90 days predicting
decline over the *next* 30 days, using the warehouse release's daily fact table. For this Week-2
framing exercise, I'm using the starter CSV's proxy label because it's what's available now and
lets me verify the framing against real numbers today — but I'm naming it as a proxy, not
pretending it's a clean future-outcome label.

In [ ]:
# No computation needed here -- Section 2 is a label-definition statement.
# The label's actual behavior in the data is verified in Section 4.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50** (with the base rate printed alongside it, always).

Why this one and not accuracy or ROC-AUC:

- **Accuracy** doesn't fit a review-queue problem — the SEO lead never sees all 30,000 pages,
  so a metric that rewards being right on the boring majority of pages nobody will look at is
  the wrong yardstick.
- **ROC-AUC** measures ranking quality across the *entire* list, including the bottom tens of
  thousands of pages that will never make anyone's worklist. It's useful for comparing models,
  but it doesn't answer the real question: "were the top K pages I handed the editor worth their
  time?"
- **Precision@50** answers exactly that: of the top 50 pages the ranking puts first, what
  fraction are actually declining-with-demand pages worth reviewing? K=50 isn't arbitrary — it
  matches the repo's own baseline pipeline (`scripts/02_baseline_score.py`,
  `scripts/03_train_model.py`), which already evaluates and reports at K=50. Reusing it keeps
  ML-03's framing consistent with the baseline and model work I'll build in ML-07/ML-08, instead
  of inventing a new K I'd have to justify twice.

"Good" means: the top-50 precision clearly beats the base rate (the declining rate across all
pages, computed below) — a small lift over the base rate would mean the ranking isn't adding much
over picking randomly.

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

base_rate = df["is_declining_label"].mean()
print(f"Base rate (declining rate across all {len(df):,} pages): {base_rate:.3f}")
print(f"-> Any ranking claiming success must beat {base_rate:.3f} at Precision@50 to be worth anything.")
print(f"-> Random top-50 picks would be right about {round(base_rate*50)} times out of 50, on average.")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis: one content page** (one row = one pseudonymized page, `content_id`), the same
grain as ML-02. Loaded below, with the columns that matter for the ranking task.

In [ ]:
cols = [
    "content_id", "client_id", "impressions_90d", "days_since_last_update",
    "avg_position", "ctr", "trend_direction", "is_declining_label",
]
print(f"{len(df):,} rows, one row per content page (content_id is unique per row)")
df[cols].head(5)

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule ("stale AND visible") is a fine *starting point* — I already used one in
`02_your_first_readable_model.ipynb`'s hand rule (`stale = days_since_last_update >= 180`,
`visible = impressions_90d >= 500`). I need to be precise about what that experiment actually
showed, because it's more nuanced than "the model wins":

- **In-sample**, my hand rule got Precision@50 = 0.680, beating my own depth-2 decision tree
  (0.600) at that same cutoff.
- On an **honest client-holdout split** (the fair comparison, since a client's pages should never
  appear in both train and test), the depth-2 tree's Precision@50 actually *dropped* to 0.520 —
  below the hand rule. So in my own small ML-02 experiment, the simple rule was not clearly beaten.

That result doesn't kill the "ML beats a fixed rule" claim — it disciplines it. The repo's full
reference pipeline (`scripts/03_train_model.py` → `outputs/model_report.md`), which trains a
properly tuned random forest on client-holdout with more features than my quick classroom tree,
gets Precision@50 = 0.740 against the same rule-based baseline's 0.240 — a real, ~3x lift, on the
same honest split. The difference between my ML-02 tree and the repo's random forest is method
and feature richness, not the split — both were evaluated on client-holdout.

So the honest claim is conditional, not absolute: **a weak, shallow model can lose to a good hand
rule** (my ML-02 result proves this concretely), but **a properly built model, given enough
relevant features, beats the rule by a wide margin** (the repo's pipeline proves this). The reasons
ML *can* win here, when built well:

1. **The signals don't move together.** `search_volume` barely correlates with actual traffic
   (`impressions_90d`, checked below) — a rule leaning on keyword volume as a proxy for importance
   ranks pages wrong. A model can weigh many weakly-correlated signals at once.
2. **Thresholds interact, they don't stack cleanly.** A stale, visible page can still be fine if
   it already ranks page-1 with a healthy CTR — the *combination* of position, CTR, and staleness
   matters, which is why the repo's random forest leans on `days_with_impressions`,
   `avg_position`, and `content_age_days` together (`outputs/model_report.md`'s feature-importance
   list), not any single hand-written condition.
3. **A rule stays static while the site doesn't.** Per the framing skill's own wording, this is a
   ranking/scoring problem because "the pattern is real but too messy to write by hand — many
   signals, tangled, shifting over time." A model can be retrained as the client/page mix shifts;
   a hand-tuned threshold has to be manually revisited.

None of this claims the model *causes* better outcomes, or that a shallow model automatically
beats a rule — only that, done properly and evaluated honestly, ranking by a fitted model
outperformed the rule baseline in this repo's own results, which is the decision-support claim my
lane can actually defend.

In [ ]:
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"Correlation between search_volume and impressions_90d: {corr:.3f}")
print("Near zero -> a rule that leans on keyword search volume as a proxy for page importance")
print("would rank pages wrong; the real traffic signal isn't captured by that single column.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.